In [2]:
import scipy.stats
from openai import OpenAI
from math import exp
import numpy as np
import json
from scipy.stats import entropy
import math
from IPython.display import display, HTML
import os
import ast
np.set_printoptions(legacy='1.25')
import pprint


In [3]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


#### GPT methods

In [4]:
def get_completion(
    messages: list[dict[str, str]],
    model: str = "gpt-4",
    max_tokens=500,
    temperature=0,
    stop=None,
    seed=123,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stop": stop,
        "seed": seed,
        "logprobs": logprobs,
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.chat.completions.create(**params)
    return completion

In [5]:
def get_ans_token_logprobs(response, anskey):
    tokenprobs = response.choices[0].logprobs.content
    i = 0
    token = tokenprobs[i].token.strip()
    #print(i, token)
    while token != anskey:
        i += 1
        token = tokenprobs[i].token.strip()
        #print('forward {}, {}'.format(i, token))

    while token != '{' and token != '{\'' and token != '{\"':
        i -= 1
        token = tokenprobs[i].token.strip()
        #print('backward {}, {}'.format(i, token))

    json_begin = i
    #print('json_begins at {}'.format(json_begin))

    while token != '}' and token != '}\'' and token != '}.' and token != '\'}' and token != '}\"':
        i += 1
        token = tokenprobs[i].token.strip()
        #print('forward {}, {}'.format(i, token))

    json_end = i
    #print(json_begin, json_end)
    response_list = tokenprobs[json_begin: json_end+1]
    response_string = "".join([o.token for o in response_list]).strip('.')
    response_dict = ast.literal_eval(response_string)

    ans_logprobs = None

    #print(str(response_dict[anskey]))
    for item in response_list:
        if item.token == str(response_dict[anskey]):
            ans_logprobs = item.top_logprobs


    return response_dict, ans_logprobs



In [6]:
def get_answer(question, prompt, anskey):
    num_next_tokens = 5
    prompt_up = prompt.format(question=question)
    #print(prompt_up)
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt_up}],
        model="gpt-4",
        logprobs=True,
        top_logprobs=num_next_tokens,
    )

    print(API_RESPONSE.choices[0].message.content)

    response_dict, ans_logprobs = get_ans_token_logprobs(API_RESPONSE, anskey)

    alters = []
    probs = []
    for item in ans_logprobs:
        linear_prob = np.round(np.exp(item.logprob)*100, 3)
        alters.append((item.token, linear_prob, item.logprob))
        probs.append(linear_prob)


    return_dict = {
        '1. prompt': prompt_up,
        '2. response': API_RESPONSE.choices[0].message.content,
        '3. response_dict': response_dict,
        '4. entropy': np.round(scipy.stats.entropy(probs, base=2)/scipy.stats.entropy([0.2, 0.2, 0.2, 0.2, 0.2], base=2), 3),
        '5. alternatives': alters
    }

    return return_dict


#### Prompts

In [11]:
basePROMPT = """You will be given a math word problem. Provide a numeric answer to the problem. MAKE SURE to only provide a numeric answer. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

In [12]:
confidencePROMPT = """You will be given a math word problem. Provide a numeric answer to the problem. MAKE SURE to only provide a numeric answer. Your confidence is how much you trust your answer to be correct and is on a scale of 1-10. Your final response should be a dictionary with key 'ans' and 'confidence'.
Problem: {question}"""

In [13]:
emptyPROMPT="""Your final response should be a dictionary with key 'ans'. Problem: {question}"""

In [14]:
cotPROMPT=""""You will be given a math word problem. Provide a numeric answer to the problem. Think step by step. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

#### Problem text

LLMs, particularly those with reasoning, are known to be good at solving algebra word problems. Here, I experiment with various levels of ambiguity in the question. The following problems are variations of the same type. The first two problems have an unambiguous answer that can be calculated in a straight forward fashion. The third problem doesn't explicitly specify which fight the question is about. The fourth problem requires additional layers of reasoning.

In [10]:
problems = {
            "straight_opt1_01": {
                    "question": "On average Joe throws 25 punches per minute. First fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds of 5 minutes. How many punches did he throw in the first fight?",
                    "correct_ans": 375,
                    "category": "unambiguous"
                },
            "straight_opt2_01":{
                    "question": "On average Joe throws 25 punches per minute. First fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds of 5 minutes. How many punches did he throw in the second fight?",
                    "correct_ans": 500,
                    "category": "unambiguous"
                },
            "ambiguous_01": {
                    "question": "On average Joe throws 25 punches per minute. First fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds of 5 minutes. How many punches did he throw in the fight?",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "total_01": {
                    "question": "On average Joe throws 25 punches per minute. First fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds of 5 minutes. How many punches did he throw in both fights?",
                    "correct_ans": 875,
                    "category": "unambiguous"
                }
}

#### Observations

We begin with the easy, unambigous questions.


In [11]:
ans = get_answer(question=problems['straight_opt1_01']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in the first fight?',
 '2. response': "{'ans': 375}",
 '3. response_dict': {'ans': 375},
 '4. entropy': 0.0,
 '5. alternatives': [('375', 100.0, 0.0),
                     ('325', 0.0, -18.034464),
                     ('25', 0.0, -18.438463),
                     ('75', 0.0, -19.007116),
                     (' ', 0.0, -19.025997)]}


In [12]:
ans = get_answer(question=problems['straight_opt2_01']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in the second '
              'fight?',
 '2. response': "{'ans': 500}",
 '3. response_dict': {'ans': 500},
 '4. entropy': 0.0,
 '5. alternatives': [('500', 99.998, -1.831257e-05),
                     ('100', 0.001, -11.603904),
                     ('200', 0.0, -12.443856),
                     ('600', 0.0, -13.573398),
                     ('125', 0.0, -13.640393)]}


For both unambiguous questions, regular prompting can generate correct answers. I report entropy of the answer which captures the degree to which the LLM is 'confused' about the answer. In both these cases, there is no confusion and hence, the entropy is 0. All the other alternatives considered by the LLM have very low probabilities. This is what we expect when reasoning works as expected.

Next, we consider the ambiguous question where the fight under consideration is not specified. We expect the entropy of the answer to be high, reflecting the LLMs is confused about the answer.

In [13]:
ans = get_answer(question=problems['ambiguous_01']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in the fight?',
 '2. response': "{'ans': 675}",
 '3. response_dict': {'ans': 675},
 '4. entropy': 0.775,
 '5. alternatives': [('675', 38.839, -0.94574225),
                     ('105', 18.935, -1.6641833),
                     ('975', 11.492, -2.1635613),
                     ('110', 3.426, -3.3737226),
                     ('950', 3.171, -3.451239)]}


As expected, the entropy of the answer in very high reflecting the confusion within LLM's inference. However, the LLM does produce a response (675, which is incorrect). This behavior is very different from how a human would respond. Because the question is ambiguous, a human would ask 'which fight are you talking about?'. The secondary question elicits further information from the speaker and constrains ambiguity. LLMs cannot modulate their response based on 'confusion' in their inference systems.

Next, I investigate if the LLM is aware of its own confusion. I asked it to report its own confidence on the answer on a scale of 1-10.

In [14]:
ans = get_answer(question=problems['ambiguous_01']['question'], prompt=confidencePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              'Your confidence is how much you trust your answer to be correct '
              'and is on a scale of 1-10. Your final response should be a '
              "dictionary with key 'ans' and 'confidence'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in the fight?',
 '2. response': "{'ans': 675, 'confidence': 10}",
 '3. response_dict': {'ans': 675, 'confidence': 10},
 '4. entropy': 0.836,
 '5. alternatives': [('675', 34.825, -1.0548251),
                     ('105', 19.296, -1.6452849),
                     ('975', 15.072, -1.89234),
                     ('112', 5.089, -2.9781382),
                     ('110', 3.69, -3.2996027)]}


In its response, the LLM claims to be highly confident (with a score of 10). It reports that it is highly confident of the answer. However, not only is the answer wrong but high answer entropy suggests that the LLM is not confident in the answer.

In [15]:
ans = get_answer(question=problems['total_01']['question'], prompt=basePROMPT, anskey='ans')
problems['total_01']
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in both fights?',
 '2. response': '{"ans": 675}',
 '3. response_dict': {'ans': 675},
 '4. entropy': 0.319,
 '5. alternatives': [('675', 81.589, -0.20347306),
                     ('975', 4.246, -3.1591253),
                     ('105', 3.245, -3.4281225),
                     ('725', 1.932, -3.9465132),
                     ('825', 1.386, -4.2786093)]}


### Math problem 2

In [8]:
thinkOutLoudPROMPT = """"You will be given a math word problem. Provide a numeric answer to the problem. Think out loud step by step. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

In [36]:
problems_part2 = {
            "straight_opt1_02": {
                    "question": "If there are 7 bottle caps in a box and Linda puts 7 more bottle caps inside, how many bottle caps are in the box?",
                    "correct_ans": 14,
                    "category": "unambiguous"
                },
            "straight_opt2_02": {
                    "question": "If there are 7 bottle caps in a box and Linda gets 7 more bottle caps, how many bottle caps are in the box?",
                    "correct_ans": 7,
                    "category": "ambiguous" # Unsure if this qualifies as ambiguous
                },
            "invalid_01": {
                    "question": "If there are 7 bottle caps in a box and Linda gets (-7) more bottle caps, how many bottle caps are in the box?",
                    "correct_ans": None,
                    "category": "invalid"
                },
            "ambiguous_01": {
                    "question": "If Linda gets 7 bottle caps and Peter gets 7 more bottle caps, how many bottle caps are in the box?",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "ambiguous_02": {
                    "question": "If Linda put 7 bottle caps in a box and Peter gets 7 more bottle caps, how many bottle caps are in the box?",
                    "correct_ans": 7,
                    "category": "ambiguous"
                },
            "ambiguous_03": {
                    "question": "If Linda put 7 bottle caps in a box and Peter gets 7 more bottle caps, how many cups are in the box?",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "ambiguous_04": {
                    "question": "If Linda put 7 bottle caps in the first box and Peter puts 10 bottle caps in the second box, how many bottle caps are in the box?",
                    "correct_ans": None,
                    "category": "ambiguous"
                }
}

In [68]:
prompt = {
            "question": "If Sherry put 7 marbles in a bowl, and Ron put 10 marbles in another bowl, how many marbles are in a bowl?",
            "correct_ans": None,
            "category": "ambiguous"
}

ans =  get_answer(question=prompt['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'ans': 17}
{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: If Sherry put 7 marbles in a bowl, and Ron put 10 '
              'marbles in another bowl, how many marbles are in a bowl?',
 '2. response': "{'ans': 17}",
 '3. response_dict': {'ans': 17},
 '4. entropy': 0.158,
 '5. alternatives': [('17', 93.572, -0.066434175),
                     ('10', 5.974, -2.8177102),
                     ('7', 0.422, -5.468576),
                     ('0', 0.009, -9.300427),
                     ('1', 0.007, -9.635306)]}


In [6]:
noRestrictionPROMPT = """You will be given a math word problem. Provide a numeric answer to the problem. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""
prompt = {
            "question": "If Sherry put 7 marbles in a bowl, and Ron put 10 marbles in another bowl, how many marbles are in a bowl?",
            "correct_ans": None,
            "category": "ambiguous"
}

ans =  get_answer(question=prompt['question'], prompt=noRestrictionPROMPT, anskey='ans')
pprint.pprint(ans)

The problem does not specify if the marbles are being combined into one bowl or if they are staying in separate bowls. Therefore, the answer could be either 7 or 10 marbles in a bowl.


IndexError: list index out of range

### Observations

In [18]:
ans = get_answer(question=problems_part2['straight_opt1_02']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: If there are 7 marbles in a bowl and Sherry puts 7 '
              'more marbles inside, how many marbles are in the box?',
 '2. response': "{'ans': 14}",
 '3. response_dict': {'ans': 14},
 '4. entropy': 0.0,
 '5. alternatives': [('14', 100.0, -4.4849444e-06),
                     ('0', 0.0, -12.6502695),
                     ('7', 0.0, -13.791778),
                     ('13', 0.0, -17.497917),
                     ('15', 0.0, -17.751488)]}


##### In the following option, I specified that Linda got 7 more bottle caps but I did not specify that it was put in the box. However, ChatGPT made the inherent assumption and with low entropy provides the wrong answer.

In [38]:
ans = get_answer(question=problems_part2['straight_opt2_02']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: If there are 7 marbles in a box and Sherry has 7 more '
              'marbles, how many marbles are in the box?',
 '2. response': "{'ans': 7}",
 '3. response_dict': {'ans': 7},
 '4. entropy': 0.007,
 '5. alternatives': [('7', 99.843, -0.0015744948),
                     ('14', 0.157, -6.454894),
                     ('8', 0.0, -15.4675),
                     ('15', 0.0, -16.374365),
                     ('13', 0.0, -16.981846)]}


##### When I ask it to think out loud, it says that Linda "adds 7 more bottle caps to the box" when the question does not say it in the first place.

In [42]:
ans = get_answer(question=problems_part2['straight_opt2_02']['question'], prompt=thinkOutLoudPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': '"You will be given a math word problem. Provide a numeric '
              'answer to the problem. Think out loud step by step. Your final '
              "response should be a dictionary with key 'ans'.\n"
              'Problem: If there are 7 marbles in a box and Sherry has 7 more '
              'marbles, how many marbles are in the box?',
 '2. response': 'The problem states that there are 7 marbles in a box. It also '
                'mentions that Sherry has 7 more marbles, but it does not '
                'specify that she puts her marbles in the box. Therefore, the '
                'number of marbles in the box remains the same, which is 7.\n'
                '\n'
                "So, the answer is {'ans': 7}.",
 '3. response_dict': {'ans': 7},
 '4. entropy': 0.0,
 '5. alternatives': [('7', 100.0, -1.147242e-06),
                     (' ', 0.0, -14.177573),
                     (' seven', 0.0, -15.982942),
                     ('07', 0.0, -16.142149),
          

##### Here I add a different person to the question. I wanted to test if ChatGPT made the previous mistake because it was the same person. But even here, GPT adds the values and has almost no confusion (even though it is wrong)

In [43]:
ans = get_answer(question=problems_part2['ambiguous_02']['question'], prompt=thinkOutLoudPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': '"You will be given a math word problem. Provide a numeric '
              'answer to the problem. Think out loud step by step. Your final '
              "response should be a dictionary with key 'ans'.\n"
              'Problem: If Linda put 7 bottle caps in a box and Peter gets 7 '
              'more bottle caps, how many bottle caps are in the box?',
 '2. response': 'First, we know that Linda put 7 bottle caps in the box. Then, '
                'Peter adds 7 more bottle caps to the box. To find the total '
                'number of bottle caps, we add the number of bottle caps Linda '
                'put in the box to the number of bottle caps Peter added. So, '
                '7 (from Linda) + 7 (from Peter) equals 14 bottle caps. \n'
                '\n'
                'So, the answer is 14 bottle caps.\n'
                '\n'
                "{'ans': 14}",
 '3. response_dict': {'ans': 14},
 '4. entropy': 0.0,
 '5. alternatives': [('14', 100.0, -1.6240566e-06)

##### In the following prompt, none of the bottle caps were actually put in the box, but ChatGPT made that assumption and gives the wrong answer.

In [32]:
ans = get_answer(question=problems_part2['ambiguous_01']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: If Linda gets 7 bottle caps and Peter gets 7 more '
              'bottle caps, how many bottle caps are in the box?',
 '2. response': "{'ans': 14}",
 '3. response_dict': {'ans': 14},
 '4. entropy': 0.001,
 '5. alternatives': [('14', 99.982, -0.00018149138),
                     ('7', 0.015, -8.836996),
                     ('0', 0.003, -10.4026985),
                     ('2', 0.0, -13.454594),
                     ('21', 0.0, -14.33108)]}


##### I hoped to achieve some differences in entropy by inserting a random object. While it still provides the wrong answer with high confidence using the base prompt, when asked to think out loud, it gives the correct answer.

In [34]:
ans = get_answer(question=problems_part2['ambiguous_03']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: If Linda put 7 bottle caps in a box and Peter gets 7 '
              'more bottle caps, how many cups are in the box?',
 '2. response': "{'ans': 7}",
 '3. response_dict': {'ans': 7},
 '4. entropy': 0.007,
 '5. alternatives': [('7', 99.847, -0.0015335473),
                     ('0', 0.137, -6.5905704),
                     ('14', 0.016, -8.749359),
                     ('8', 0.0, -15.892533),
                     ('9', 0.0, -16.378593)]}


In [35]:
ans = get_answer(question=problems_part2['ambiguous_03']['question'], prompt=thinkOutLoudPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': '"You will be given a math word problem. Provide a numeric '
              'answer to the problem. Think out loud step by step. Your final '
              "response should be a dictionary with key 'ans'.\n"
              'Problem: If Linda put 7 bottle caps in a box and Peter gets 7 '
              'more bottle caps, how many cups are in the box?',
 '2. response': 'The problem does not provide information about any cups being '
                'placed in the box. Therefore, the answer is 0.\n'
                '\n'
                'So, the final response would be:\n'
                '\n'
                "{'ans': 0}",
 '3. response_dict': {'ans': 0},
 '4. entropy': 0.0,
 '5. alternatives': [('0', 99.999, -7.58424e-06),
                     (' ', 0.001, -11.9128685),
                     ('<|end|>', 0.0, -14.408349),
                     ('00', 0.0, -15.455267),
                     ('000', 0.0, -18.012653)]}


##### I tried to replicate the ambiguity in the first problem by introducing two boxes and asking it to tell me the number of bottle caps in "the box" without specifying which one. But it seems to provide the wrong answer with low entropy.

In [37]:
ans = get_answer(question=problems_part2['ambiguous_04']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: If Sherry put 7 marbles in the first bowl and Ron has '
              '10 marbles in the second bowl, how many marbles are in the '
              'bowl?',
 '2. response': '{"ans": 17}',
 '3. response_dict': {'ans': 17},
 '4. entropy': 0.0,
 '5. alternatives': [('17', 100.0, -1.3856493e-06),
                     ('7', 0.0, -14.181243),
                     ('10', 0.0, -16.015608),
                     ('0', 0.0, -16.453712),
                     ('15', 0.0, -16.740911)]}


##### Here, I give it an invalid option. I say that Linda got -7 more bottle caps. This is where a human would technically say this is not possible. But GPT performs the math operation by default. It provides a reasoning even when asked to think out loud.

In [40]:
ans = get_answer(question=problems_part2['invalid_01']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: If there are 7 bottle caps in a box and Linda gets '
              '(-7) more bottle caps, how many bottle caps are in the box?',
 '2. response': "{'ans': 0}",
 '3. response_dict': {'ans': 0},
 '4. entropy': 0.009,
 '5. alternatives': [('0', 99.801, -0.0019912054),
                     ('7', 0.198, -6.225422),
                     ('14', 0.001, -11.469793),
                     ('1', 0.0, -16.589758),
                     (' ', 0.0, -16.859762)]}


In [36]:
ans = get_answer(question=problems_part2['invalid_01']['question'], prompt=thinkOutLoudPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': '"You will be given a math word problem. Provide a numeric '
              'answer to the problem. Think out loud step by step. Your final '
              "response should be a dictionary with key 'ans'.\n"
              'Problem: If there are 7 bottle caps in a box and Linda gets '
              '(-7) more bottle caps, how many bottle caps are in the box?',
 '2. response': 'First, we start with the number of bottle caps in the box, '
                'which is 7. \n'
                '\n'
                'Then, we add the number of bottle caps Linda gets, which is '
                '-7. \n'
                '\n'
                'Adding a negative number is the same as subtracting that '
                'number. So, we subtract 7 from 7. \n'
                '\n'
                '7 - 7 equals 0. \n'
                '\n'
                'So, there are 0 bottle caps in the box.\n'
                '\n'
                "The final answer is {'ans': 0}.",
 '3. response_dict': {'ans

### Prompts for common sense reasoning

In [69]:
baseCommonPROMPT = """You will be given a commonsense reasoning problem. You will then be provided possible answer options labeled with letters following the question mark. Provide the letter assigned to the correct answer to the problem. MAKE SURE to only provide a letter as an answer. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

In [40]:
multipleAnswerPROMPT =  """You will be given a commonsense reasoning problem. You will then be provided possible answer options labeled with letters following the question mark. Provide the letter assigned to the correct answer to the problem. There may be multiple answers. Provide a reasoning for why you made the choice. MAKE SURE to only provide one letter as an answer. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

In [60]:
multipleChoicesPROMPT =  """You will be given a commonsense reasoning problem. You will then be provided possible answer options labeled with letters following the question mark. Provide the letter assigned to the correct answer to the problem. There may be multiple answers. Provide a reasoning for why you made the choices including probabilities for the remaining probable answers. MAKE SURE to only provide one letter as an answer. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

In [1]:
baseCommonReasoningPROMPT = """You will be given a commonsense reasoning problem. You will then be provided possible answer options labeled with letters following the question mark. Provide the letter assigned to the correct answer to the problem. Provide a reasoning for why you made the choice. Your final response should be a dictionary with keys 'ans' and 'reasoning'
Problem: {question}"""

### Problem text

In [92]:
problems_common = {
            "straight_opt1_01": {
                    "question": "Sammy wanted to go to where the people were.  Where might he go? A: Race track, B: Populated areas, C: the desert, D: apartment, E: roadblock ",
                    "correct_ans": "B",
                    "category": "unambiguous"
                },

            "ambiguous_01": {
                    "question": "Sammy wanted to go to where the people were.  Where might he go? A: Restaurant, B: New York, C: the desert, D: gym in the morning, E: roadblock ",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "ambiguous_02": {
                    "question": "Why would you be able to see through a door? A: Door is open, B: Door is made of glass, C: xray vision, D: door is made of wood, E: Door is made of plastic ",
                    "correct_ans": None,
                    "category": "ambiguous"
            },
            "ambiguous_03": {
                    "question": "Where can you go to have a person assist you in mailing a package? A: USPS, B: Fedex, C: UPS",
                    "correct_ans": None,
                    "category": "ambiguous"
            },
            "ambiguous_04": {
                    "question": "Sammy wanted to go to where the people were.  Where might he go? A: roadblock, B: New York, C: the desert, D: gym in the morning, E: Restaurant, F: Restaurant ",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "ambiguous_05": {
                    "question": "Sammy wanted to go to where the people were.  Where might he go? A: Restaurant, B:  Restaurant, C: the desert, D: gym in the morning, E: roadblock, F: New York",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "ambiguous_06": {
                    "question": "Sammy wanted to go to where the people were.  Where might he go? A: Chain, B:  tree, C: table, D: computer, E: chair, F: washing machine",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "ambiguous_07": {
                    "question": "Sammy wanted to go to where the people were eating.  Where might he go? B:  tree, C: table, D: computer, E: chair, F: washing machine",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "ambiguous_08": {
                    "question": "Sammy wanted to go to where the people were eating.  Where might he go? D: cellphone, E: computer, F: washing machine, G: pen",
                    "correct_ans": None,
                    "category": "ambiguous"
                }
}

### Observations for common sense data



In [19]:
ans = get_answer(question=problems_common['straight_opt1_01']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Race track, B: Populated areas, C: the desert, '
              'D: apartment, E: roadblock ',
 '2. response': "{'ans': 'B'}",
 '3. response_dict': {'ans': 'B'},
 '4. entropy': 0.0,
 '5. alternatives': [('B', 100.0, -1.9361265e-07),
                     ('A', 0.0, -16.041931),
                     ('D', 0.0, -17.027851),
                     (' B', 0.0, -19.160372),
                     ('E', 0.0, -20.221634)]}


In [32]:
ans = get_answer(question=problems_common['ambiguous_01']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Restaurant, B: New York, C: the desert, D: gym '
              'in the morning, E: roadblock ',
 '2. response': "{'ans': 'A'}",
 '3. response_dict': {'ans': 'A'},
 '4. entropy': 0.0,
 '5. alternatives': [('A', 99.997, -2.8444882e-05),
                     ('D', 0.001, -11.132879),
                     ('B', 0.001, -11.193483),
                     (' A', 0.0, -17.485836),
                     ('E', 0.0, -17.773832)]}


In [61]:
ans = get_answer(question=problems_common['ambiguous_01']['question'], prompt=multipleChoicesPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choices including '
              'probabilities for the remaining probable answers. MAKE SURE to '
              'only provide one letter as an answer. Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Restaurant, B: New York, C: the desert, D: gym '
              'in the morning, E: roadblock ',
 '2. response': "{'ans': 'A'}\n"
                '\n'
                'Reasoning: The question asks where Sammy might go if he '
                'wanted to be where the people were. The most 

In [37]:
ans = get_answer(question=problems_common['ambiguous_02']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Why would you be able to see through a door? A: Door '
              'is open, B: Door is made of glass, C: xray vision, D: door is '
              'made of wood, E: Door is made of plastic ',
 '2. response': "{'ans': 'B'}",
 '3. response_dict': {'ans': 'B'},
 '4. entropy': 0.0,
 '5. alternatives': [('B', 100.0, -7.89631e-07),
                     ('A', 0.0, -14.093112),
                     (' B', 0.0, -17.349575),
                     ('C', 0.0, -18.364323),
                     ('D', 0.0, -18.935795)]}


In [41]:
ans = get_answer(question=problems_common['ambiguous_02']['question'], prompt=multipleAnswerPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choice. MAKE SURE to '
              'only provide one letter as an answer. Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Why would you be able to see through a door? A: Door '
              'is open, B: Door is made of glass, C: xray vision, D: door is '
              'made of wood, E: Door is made of plastic ',
 '2. response': "{'ans': 'B'} Reasoning: A door made of glass would allow you "
                'to see through it. While an open door or xray vision could '
                'technically allow you to see through a door, the question '
                'seem

In [47]:
ans = get_answer(question=problems_common['ambiguous_03']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Where can you go to have a person assist you in '
              'mailing a package? A: USPS, B: Fedex, C: UPS',
 '2. response': "{'ans': 'A'}",
 '3. response_dict': {'ans': 'A'},
 '4. entropy': 0.094,
 '5. alternatives': [('A', 96.585, -0.03474311),
                     ('D', 3.384, -3.386195),
                     ('C', 0.023, -8.396432),
                     ('All', 0.004, -10.193547),
                     ('B', 0.004, -10.213874)]}


In [48]:
ans = get_answer(question=problems_common['ambiguous_03']['question'], prompt=multipleAnswerPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choice. MAKE SURE to '
              'only provide one letter as an answer. Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Where can you go to have a person assist you in '
              'mailing a package? A: USPS, B: Fedex, C: UPS',
 '2. response': "{'ans': 'A'}\n"
                '\n'
                'Reasoning: USPS is a government-run postal service where you '
                'can go to have a person assist you in mailing a package. '
                'Although Fedex and UPS also offer similar services, the '
                'question asks for only one answer.'

In [55]:
ans = get_answer(question=problems_common['ambiguous_03']['question'], prompt=multipleChoicesPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choices. Give '
              'probabilities for the remaining probable answers. MAKE SURE to '
              'only provide one letter as an answer.   Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Where can you go to have a person assist you in '
              'mailing a package? A: USPS, B: Fedex, C: UPS',
 '2. response': "{'ans': 'A', 'reasoning': 'All of the options provided, USPS, "
                'Fedex, and UPS, are places where you can go to have a person '
                'assist you in mailing a package. However, since the '
                'instruction i

In [68]:
ans = get_answer(question=problems_common['ambiguous_04']['question'], prompt=multipleAnswerPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choice. MAKE SURE to '
              'only provide one letter as an answer. Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: roadblock, B: New York, C: the desert, D: gym '
              'in the morning, E: Restaurant, F: Restaurant ',
 '2. response': "{'ans': 'B'} \n"
                '\n'
                'Reasoning: New York is a place where there are a lot of '
                'people. While a gym, restaurant, or roadblock might have '
                "people, it's not guaranteed. T

In [70]:
ans = get_answer(question=problems_common['ambiguous_04']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: roadblock, B: New York, C: the desert, D: gym '
              'in the morning, E: Restaurant, F: Restaurant ',
 '2. response': "{'ans': 'B'}",
 '3. response_dict': {'ans': 'B'},
 '4. entropy': 0.043,
 '5. alternatives': [('B', 98.857, -0.011497984),
                     ('E', 0.927, -4.6814084),
                     ('F', 0.198, -6.2266927),
                     ('D', 0.019, -8.574128),
                     ('C', 0.0, -14.199243)]}


In [72]:
ans = get_answer(question=problems_common['ambiguous_05']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Restaurant, B:  Restaurant, C: the desert, D: '
              'gym in the morning, E: roadblock, F: New York',
 '2. response': "{'ans': 'A'}",
 '3. response_dict': {'ans': 'A'},
 '4. entropy': 0.011,
 '5. alternatives': [('A', 99.771, -0.002291806),
                     ('B', 0.127, -6.665019),
                     ('F', 0.081, -7.1229296),
                     ('D', 0.021, -8.478255),
                     ('C', 0.0, -16.712936)]}


In [75]:
ans = get_answer(question=problems_common['ambiguous_06']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Chain, B:  tree, C: table, D: computer, E: '
              'chair, F: washing machine',
 '2. response': "{'ans': 'A'}",
 '3. response_dict': {'ans': 'A'},
 '4. entropy': 0.002,
 '5. alternatives': [('A', 99.968, -0.00032271104),
                     ('C', 0.031, -8.067678),
                     ('F', 0.0, -12.48242),
                     ('D', 0.0, -13.084791),
                     ('B', 0.0, -13.24234)]}


In [77]:
ans = get_answer(question=problems_common['ambiguous_06']['question'], prompt=baseCommonReasoningPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. Provide a reasoning for why you '
              'made the choice. MAKE SURE to only provide a letter as an '
              'answer. Your final response should be a dictionary with key '
              "'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Chain, B:  tree, C: table, D: computer, E: '
              'chair, F: washing machine',
 '2. response': "{'ans': 'A'} \n"
                '\n'
                'Reasoning: A chain usually refers to a chain of stores or '
                'restaurants, which are places where people often gather. The '
                "other options are inanimate objects that don't necessarily "
                'im

In [80]:
ans = get_answer(question=problems_common['ambiguous_07']['question'], prompt=baseCommonReasoningPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. Provide a reasoning for why you '
              'made the choice. MAKE SURE to only provide a letter as an '
              'answer. Your final response should be a dictionary with key '
              "'ans'.\n"
              'Problem: Sammy wanted to go to where the people were eating.  '
              'Where might he go? B:  tree, C: table, D: computer, E: chair, '
              'F: washing machine',
 '2. response': "{'ans': 'C'} \n"
                '\n'
                'Reasoning: The most common place where people eat is at a '
                'table. Therefore, if Sammy wanted to go where people were '
                'eating, he would most likely go to a table. The other options '
                'do no

In [93]:
ans = get_answer(question=problems_common['ambiguous_08']['question'], prompt=baseCommonReasoningPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. Provide a reasoning for why you '
              'made the choice. MAKE SURE to only provide a letter as an '
              'answer. Your final response should be a dictionary with key '
              "'ans'.\n"
              'Problem: Sammy wanted to go to where the people were eating.  '
              'Where might he go? D: cellphone, E: computer, F: washing '
              'machine, G: pen',
 '2. response': "{'ans': 'E'} \n"
                '\n'
                'Reasoning: None of the options provided are places where '
                'people typically eat. However, considering the current '
                'digital age, people might be eating in front of their '
                'computers while working o

### Problem - Estimating time to eat a jar of candy


In [14]:
prompt1 = {
            "question": "Two friends are eating a jar full of candies. Had P eaten alone, it would have taken him 10 minutes to finish the candies in the jar. Had Q eaten alone, it would have taken her 5 minutes to finish half the jar. Since both of them are eating simultaneously, how many minutes would it take them to empty the jar?, A)4, B)5, C)6, D)7, E)8",
            "correct_ans": 5,
            "category": "straight"
}
pprint.pprint(get_answer(question=prompt1["question"], prompt=baseCommonReasoningPROMPT, anskey='ans'))

{'ans': 'B', 'reasoning': "P can eat the whole jar in 10 minutes, so he can eat half the jar in 5 minutes. Q can also eat half the jar in 5 minutes. Therefore, if they eat together, they can finish the whole jar in 5 minutes because they are eating simultaneously and their eating speeds add up."}
{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. Provide a reasoning for why you '
              'made the choice. Your final response should be a dictionary '
              "with keys 'ans' and 'reasoning'\n"
              'Problem: Two friends are eating a jar full of candies. Had P '
              'eaten alone, it would have taken him 10 minutes to finish the '
              'candies in the jar. Had Q eaten alone, it would have taken her '
              '5 minu

In [17]:
prompt1a = {
    "question": "Two friends are eating a jar full of candies. Had P eaten alone, it would have taken him 10 minutes to finish the candies in the jar. Had Q eaten alone, it would have taken her 5 minutes to finish half the jar. Since both of them are eating simultaneously, how many minutes would it take them to empty the jar?, A)4 B)10 C)6 D)7 E)8",
    "correct_ans": None,
    "category": "ambiguous"
}
pprint.pprint(get_answer(question=prompt1a["question"], prompt=baseCommonReasoningPROMPT, anskey='ans'))

{'ans': 'A', 'reasoning': "P can eat the whole jar in 10 minutes, so he can eat half the jar in 5 minutes. Q can also eat half the jar in 5 minutes. Therefore, if they eat together, they can finish the whole jar in 5 minutes. However, since they are eating simultaneously, they will finish faster. So, it will take less than 5 minutes. The only option less than 5 is 4. So, the answer is A)4."}
{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. Provide a reasoning for why you '
              'made the choice. Your final response should be a dictionary '
              "with keys 'ans' and 'reasoning'\n"
              'Problem: Two friends are eating a jar full of candies. Had P '
              'eaten alone, it would have taken him 10 minutes to finish the '
    

In [19]:
prompt1b = {
    "question": "Two friends are eating a jar full of candies. Had P eaten alone, it would have taken him 10 minutes to finish the candies in the jar. Had Q eaten alone, it would have taken her 5 minutes to finish half the jar. Since both of them are eating simultaneously, how many minutes would it take them to empty the jar?, A)11, B)10, C)6, D)7, E)8",
    "correct_ans": None,
    "category": "ambiguous"
}
pprint.pprint(get_answer(question=prompt1b["question"], prompt=baseCommonReasoningPROMPT, anskey='ans'))

{'ans': 'C', 'reasoning': "P can eat the whole jar in 10 minutes, so he can eat half the jar in 5 minutes. Q can also eat half the jar in 5 minutes. Therefore, if they eat together, they can finish the whole jar in 5 minutes because they are eating simultaneously and each can finish half the jar in that time."}
{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. Provide a reasoning for why you '
              'made the choice. Your final response should be a dictionary '
              "with keys 'ans' and 'reasoning'\n"
              'Problem: Two friends are eating a jar full of candies. Had P '
              'eaten alone, it would have taken him 10 minutes to finish the '
              'candies in the jar. Had Q eaten alone, it would have taken her '
      

In [18]:
prompt1c = {
    "question": "Two friends are eating a jar full of candies. Had P eaten alone, it would have taken him 10 minutes to finish the candies in the jar. Had Q eaten alone, it would have taken her 5 minutes to finish half the jar. Since both of them are eating simultaneously, how many minutes would it take them to empty the jar?, A)apple, B)orange, C)pineapple, D)tomato, E)potato",
    "correct_ans": None,
    "category": "ambiguous"
}

pprint.pprint(get_answer(question=prompt1c["question"], prompt=baseCommonReasoningPROMPT, anskey='ans'))

{'ans': 'E', 'reasoning': "None of the options A-D make sense as they are all fruits and not related to the time it would take to eat the candies. The only option that could potentially make sense is E) potato, but this is also not a measure of time. The question does not provide a correct answer option as it asks for a time duration, not a type of fruit or vegetable. Therefore, the answer is E) potato by default, even though it does not logically answer the question."}
{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. Provide a reasoning for why you '
              'made the choice. Your final response should be a dictionary '
              "with keys 'ans' and 'reasoning'\n"
              'Problem: Two friends are eating a jar full of candies. Had P '
   

In [20]:
generalAnswerPROMPT = """You will be given a commonsense reasoning problem. Provide the correct answer to the problem. Provide a reasoning for why you made the choice. Your final response should be a dictionary with keys 'ans' and 'reasoning'
Problem: {question}"""
prompt1d = {
    "question": "Two friends are eating a jar full of candies. Had P eaten alone, it would have taken him 10 minutes to finish the candies in the jar. Had Q eaten alone, it would have taken her 5 minutes to finish half the jar. Since both of them are eating simultaneously, how many minutes would it take them to empty the jar?",
    "correct_ans": None,
    "category": "ambiguous"
}
pprint.pprint(get_answer(question=prompt1d["question"], prompt=generalAnswerPROMPT, anskey='ans'))

{'ans': 4, 'reasoning': "P can eat the whole jar in 10 minutes, so he can eat half the jar in 5 minutes. Q can also eat half the jar in 5 minutes. Therefore, if they eat together, they can finish the whole jar in 5 minutes. However, since they are eating simultaneously, they are effectively doubling their eating speed, so it would take them half the time, or 4 minutes, to finish the jar."}
{'1. prompt': 'You will be given a commonsense reasoning problem. Provide the '
              'correct answer to the problem. Provide a reasoning for why you '
              'made the choice. Your final response should be a dictionary '
              "with keys 'ans' and 'reasoning'\n"
              'Problem: Two friends are eating a jar full of candies. Had P '
              'eaten alone, it would have taken him 10 minutes to finish the '
              'candies in the jar. Had Q eaten alone, it would have taken her '
              '5 minutes to finish half the jar. Since both of them are eating '
  